# Atividade — Classificação com K-Means e Aprendizado Supervisionado

**Base de dados:** Spambase 

**Enunciado:**

Usando o scikit-learn:

1. Execute o k-means 100 vezes com k ∈ {2, 3, 4, 5}. Para cada k (executado 100 vezes) selecione o melhor resultado segundo a função objetivo. Para cada k, calcule a silhueta (Sil). Faça o plot Sil × k para k ∈ {2, 3, 4, 5} e escolha o número de grupos: k* = arg max_k Sil(k).

2. Assuma que os clusters são as verdadeiras classes. Desconsidere a variável resposta original e acrescente uma nova variável resposta onde os valores possíveis para os exemplos são entre 1 e k*. Use os 5 (cinco) classificadores abaixo:
   - i) Árvores de decisão;
   - ii) Bayesiano ingênuo;
   - iii) Regressão logística;
   - iv) k-vizinhos (Use conjunto de validação para fixar o par (número de vizinhos k, distância));
   - v) Random forest.

3. Separe de maneira aleatória e estratificada (proporcional ao número de exemplos de cada classe) os dados em treinamento, teste e, se necessário, validação. Sugestão: 80% para treinamento + validação e 20% para teste.

4. Treine o classificador no conjunto de treinamento. Teste o classificador no conjunto de teste. Se o modelo possuir hiperparâmetros, use validação cruzada para fixar os hiperparâmetros. O conjunto de teste não pode ser usado no treinamento e na escolha dos valores dos hiperparâmetros. Tentar melhorar cada um dos classificadores escolhido, procurando os melhores hiperparâmetros para alcançar a melhor precision (precisão), cobertura (recall) e f1-score no conjunto de teste.

5. Para cada métrica de avaliação, plot a curva de aprendizagem para os classificadores com a melhor configuração de hiperparâmetros. Mais precisamente, considere conjuntos de treinamento e teste de (5%, 95%) a (95%, 5%) do conjunto original de dados, com passo de 5% (usando amostragem estratificada). Para cada par de conjuntos de treinamento e teste, calcule as métricas de avaliação em cada um desses conjuntos. Comente.

# Etapa 1 — K-Means e Escolha de k*

Objetivo: executar o K-Means 100 vezes para cada k ∈ {2, 3, 4, 5}, selecionar a melhor execução pela função objetivo (inércia), calcular a silhueta de cada k e determinar k* = argmax Sil(k).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

os.makedirs('results', exist_ok=True)

## 1.1 Carregamento e pré-processamento dos dados

In [23]:
# Nomes das 57 features + label conforme spambase.names
feature_names = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d',
    'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet',
    'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free',
    'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650',
    'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857',
    'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project',
    'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_semicolon', 'char_freq_parenthesis', 'char_freq_bracket',
    'char_freq_exclamation', 'char_freq_dollar', 'char_freq_hash',
    'capital_run_length_average', 'capital_run_length_longest',
    'capital_run_length_total'
]

col_names = feature_names + ['is_spam']

df = pd.read_csv('spambase.data', header=None, names=col_names)
print(f'Amostras: {len(df)}')
print(f'Features: {len(feature_names)}')
print(f'Distribuição do label original:')
print(df['is_spam'].value_counts())
df.head()

Amostras: 4601
Features: 57
Distribuição do label original:
is_spam
0    2788
1    1813
Name: count, dtype: int64


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_semicolon,char_freq_parenthesis,char_freq_bracket,char_freq_exclamation,char_freq_dollar,char_freq_hash,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


In [24]:
# Separar features (sem o label original) e normalizar
X = df[feature_names].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Shape dos dados normalizados: {X_scaled.shape}')

Shape dos dados normalizados: (4601, 57)


## 1.2 K-Means: 100 execuções por valor de k

Para cada k, rodamos 100 vezes com inicializações aleatórias diferentes (`n_init=1`, variando a `random_state`). Guardamos o modelo com menor inércia (função objetivo do K-Means = soma das distâncias quadráticas intra-cluster).

In [25]:
k_values = [2, 3, 4, 5]
n_runs = 100

best_models = {}   # k -> melhor KMeans
best_inertias = {} # k -> menor inércia
best_labels = {}   # k -> labels do melhor modelo

for k in k_values:
    best_inertia = np.inf
    best_model = None
    
    for run in range(n_runs):
        km = KMeans(
            n_clusters=k,
            n_init=1,           # uma inicialização por execução
            max_iter=300,
            random_state=run    # seed diferente a cada execução
        )
        km.fit(X_scaled)
        
        if km.inertia_ < best_inertia:
            best_inertia = km.inertia_
            best_model = km
    
    best_models[k] = best_model
    best_inertias[k] = best_inertia
    best_labels[k] = best_model.labels_
    
    print(f'k={k}  |  Melhor inércia: {best_inertia:.2f}')

k=2  |  Melhor inércia: 242529.07
k=3  |  Melhor inércia: 231547.96
k=4  |  Melhor inércia: 223997.44
k=5  |  Melhor inércia: 218500.30


## 1.3 Cálculo da Silhueta para cada k

In [26]:
silhouettes = {}

for k in k_values:
    sil = silhouette_score(X_scaled, best_labels[k])
    silhouettes[k] = sil
    print(f'k={k}  |  Silhueta: {sil:.4f}')

# Determinar k*
k_star = max(silhouettes, key=silhouettes.get)
print(f'\nk* = {k_star}  (silhueta = {silhouettes[k_star]:.4f})')

k=2  |  Silhueta: 0.6596
k=3  |  Silhueta: 0.1251
k=4  |  Silhueta: 0.1582
k=5  |  Silhueta: 0.1610

k* = 2  (silhueta = 0.6596)


## 1.4 Plot Silhueta × k

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ks = list(silhouettes.keys())
sils = list(silhouettes.values())

ax.plot(ks, sils, 'o-', color='#2563eb', linewidth=2, markersize=8)

# Destaca k*
ax.plot(k_star, silhouettes[k_star], 's', color='#dc2626', markersize=14,
        zorder=5, label=f'k* = {k_star} (Sil = {silhouettes[k_star]:.4f})')

ax.set_xlabel('Número de clusters (k)', fontsize=12)
ax.set_ylabel('Silhueta média', fontsize=12)
ax.set_title('Silhueta × k — K-Means no Spambase', fontsize=14)
ax.set_xticks(ks)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/01_silhueta_vs_k.png', dpi=150)
plt.show()

print(f'\nConclusão: k* = {k_star}')

## 1.5 Salvar labels do melhor agrupamento para as próximas etapas

Os rótulos gerados pelo K-Means com k* substituirão a variável resposta original nas etapas seguintes. Salvamos em disco para não precisar re-executar o clustering.

In [ ]:
# Labels do clustering com k* (valores de 0 a k*-1)
# Convertemos para 1..k* conforme enunciado
cluster_labels = best_labels[k_star] + 1

df_clusters = df[feature_names].copy()
df_clusters['cluster'] = cluster_labels

df_clusters.to_csv('results/02_spambase_clustered.csv', index=False)
print(f'Salvo results/02_spambase_clustered.csv com {len(df_clusters)} amostras e k*={k_star} clusters')
print(f'Distribuição dos clusters:')
print(df_clusters['cluster'].value_counts().sort_index())

In [ ]:
# Salvar também o scaler e o k* para uso posterior
import pickle

with open('results/03_etapa1_resultado.pkl', 'wb') as f:
    pickle.dump({
        'k_star': k_star,
        'silhouettes': silhouettes,
        'best_inertias': best_inertias,
        'scaler': scaler,
        'best_model': best_models[k_star]
    }, f)

print('Resultados da Etapa 1 salvos em results/03_etapa1_resultado.pkl')

---
# Etapa 2 — Classificadores sobre os clusters

Descartamos o label original (`is_spam`) e usamos os clusters do K-Means como classes. Com k\*=2, temos classificação binária (cluster 1 vs cluster 2, com desbalanceamento forte: ~99.3% vs ~0.7%).

Classificadores:
1. Árvore de decisão
2. Naive Bayes (Gaussiano)
3. Regressão logística
4. KNN (k-vizinhos) — número de vizinhos e distância via validação cruzada
5. Random Forest

Separação estratificada 80/20, busca de hiperparâmetros via GridSearchCV (5-fold estratificado) apenas no conjunto de treino.

In [30]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

## 2.1 Preparação dos dados com a nova variável resposta

In [31]:
# Usar os dados já em memória da Etapa 1 (sem recarregar do CSV)
# X_scaled vem da normalização feita na Etapa 1
# cluster_labels vem do K-Means com k*

X_cls = X_scaled  # features normalizadas (57 colunas)
y_cls = cluster_labels  # labels 1..k*

print(f'Features: {X_cls.shape[1]}')
print(f'Amostras: {X_cls.shape[0]}')
print(f'Classes (clusters): {np.unique(y_cls)}')
print(f'Distribuição:')
for c in np.unique(y_cls):
    print(f'  Cluster {c}: {np.sum(y_cls == c)} ({100*np.sum(y_cls == c)/len(y_cls):.1f}%)')

Features: 57
Amostras: 4601
Classes (clusters): [1 2]
Distribuição:
  Cluster 1: 34 (0.7%)
  Cluster 2: 4567 (99.3%)


## 2.2 Separação treino/teste (estratificada, 80/20)

A estratificação mantém a proporção entre clusters nos dois conjuntos. A validação cruzada (5-fold estratificado) será aplicada dentro do treino para busca de hiperparâmetros — o teste fica isolado.

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')
print(f'\nDistribuição no treino:')
for c in np.unique(y_train):
    print(f'  Cluster {c}: {np.sum(y_train == c)}')
print(f'Distribuição no teste:')
for c in np.unique(y_test):
    print(f'  Cluster {c}: {np.sum(y_test == c)}')

Treino: 3680 amostras
Teste:  921 amostras

Distribuição no treino:
  Cluster 1: 27
  Cluster 2: 3653
Distribuição no teste:
  Cluster 1: 7
  Cluster 2: 914


## 2.3 Classificadores e grids de hiperparâmetros

Cada classificador tem um grid de hiperparâmetros explorado via `GridSearchCV` com 5-fold estratificado. O scoring é `f1_weighted` para compensar o desbalanceamento entre clusters.

In [33]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

classifiers = {
    'Árvore de Decisão': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [3, 5, 10, 20, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    },
    'Naive Bayes': {
        'model': GaussianNB(),
        'params': {
            'var_smoothing': np.logspace(-12, -6, 7)
        }
    },
    'Regressão Logística': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'class_weight': [None, 'balanced']
        }
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21],
            'metric': ['euclidean', 'manhattan', 'chebyshev'],
            'weights': ['uniform', 'distance']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 5],
            'class_weight': [None, 'balanced']
        }
    }
}

print('Classificadores definidos:')
for name, cfg in classifiers.items():
    n_combos = 1
    for v in cfg['params'].values():
        n_combos *= len(v)
    print(f'  {name}: {n_combos} combinações de hiperparâmetros')

Classificadores definidos:
  Árvore de Decisão: 160 combinações de hiperparâmetros
  Naive Bayes: 7 combinações de hiperparâmetros
  Regressão Logística: 40 combinações de hiperparâmetros
  KNN: 48 combinações de hiperparâmetros
  Random Forest: 216 combinações de hiperparâmetros


## 2.4 Treinamento com busca de hiperparâmetros (GridSearchCV)

O `GridSearchCV` testa todas as combinações no conjunto de treino via 5-fold CV estratificado. O conjunto de teste não participa da seleção de hiperparâmetros.

In [34]:
results = {}

for name, cfg in classifiers.items():
    print(f'\n{"="*60}')
    print(f'Treinando: {name}')
    print(f'{"="*60}')
    
    grid = GridSearchCV(
        estimator=cfg['model'],
        param_grid=cfg['params'],
        cv=cv_strategy,
        scoring='f1_weighted',
        n_jobs=-1,
        refit=True
    )
    grid.fit(X_train, y_train)
    
    y_pred = grid.predict(X_test)
    
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results[name] = {
        'best_params': grid.best_params_,
        'best_cv_score': grid.best_score_,
        'best_model': grid.best_estimator_,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_pred': y_pred
    }
    
    print(f'Melhores hiperparâmetros: {grid.best_params_}')
    print(f'Melhor F1 na CV: {grid.best_score_:.4f}')
    print(f'\nResultados no teste:')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-score:  {f1:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, zero_division=0))


Treinando: Árvore de Decisão
Melhores hiperparâmetros: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Melhor F1 na CV: 0.9990

Resultados no teste:
  Precision: 0.9990
  Recall:    0.9989
  F1-score:  0.9990

Classification Report:
              precision    recall  f1-score   support

           1       0.88      1.00      0.93         7
           2       1.00      1.00      1.00       914

    accuracy                           1.00       921
   macro avg       0.94      1.00      0.97       921
weighted avg       1.00      1.00      1.00       921


Treinando: Naive Bayes
Melhores hiperparâmetros: {'var_smoothing': np.float64(1e-07)}
Melhor F1 na CV: 0.9658

Resultados no teste:
  Precision: 0.9919
  Recall:    0.9305
  F1-score:  0.9576

Classification Report:
              precision    recall  f1-score   support

           1       0.09      0.86      0.16         7
           2       1.00      0.93      0.96       914

    accuracy         

## 2.5 Comparação dos classificadores

In [35]:
# Tabela comparativa
df_results = pd.DataFrame({
    'Classificador': list(results.keys()),
    'Precision': [results[n]['precision'] for n in results],
    'Recall': [results[n]['recall'] for n in results],
    'F1-score': [results[n]['f1'] for n in results],
    'F1 CV (treino)': [results[n]['best_cv_score'] for n in results]
}).sort_values('F1-score', ascending=False)

print('Comparação final no conjunto de teste:')
print(df_results.to_string(index=False))

Comparação final no conjunto de teste:
      Classificador  Precision   Recall  F1-score  F1 CV (treino)
      Random Forest   1.000000 1.000000  1.000000        0.999713
                KNN   1.000000 1.000000  1.000000        1.000000
  Árvore de Decisão   0.999050 0.998914  0.998950        0.998976
Regressão Logística   0.999050 0.998914  0.998950        1.000000
        Naive Bayes   0.991896 0.930510  0.957635        0.965781


In [ ]:
# Gráfico comparativo
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(results))
names = list(results.keys())
width = 0.25

precs = [results[n]['precision'] for n in names]
recs = [results[n]['recall'] for n in names]
f1s = [results[n]['f1'] for n in names]

ax.bar(x_pos - width, precs, width, label='Precision', color='#2563eb')
ax.bar(x_pos, recs, width, label='Recall', color='#16a34a')
ax.bar(x_pos + width, f1s, width, label='F1-score', color='#dc2626')

ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_title('Comparação dos classificadores — Conjunto de teste')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/04_comparacao_classificadores.png', dpi=150)
plt.show()

In [ ]:
# Salvar resultados para as próximas etapas
with open('results/05_etapa2_resultado.pkl', 'wb') as f:
    pickle.dump({
        'results': {name: {
            'best_params': results[name]['best_params'],
            'best_model': results[name]['best_model'],
            'precision': results[name]['precision'],
            'recall': results[name]['recall'],
            'f1': results[name]['f1']
        } for name in results},
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'scaler': scaler
    }, f)

print('Resultados da Etapa 2 salvos em results/05_etapa2_resultado.pkl')
print('\nMelhores hiperparâmetros por classificador:')
for name in results:
    print(f'\n  {name}:')
    for param, val in results[name]['best_params'].items():
        print(f'    {param} = {val}')

---
# Etapa 3 — Curvas de aprendizagem

Para cada classificador (com os melhores hiperparâmetros encontrados na Etapa 2), variamos a proporção treino/teste de (5%, 95%) até (95%, 5%) com passo de 5%, sempre com amostragem estratificada. Em cada split, calculamos precision, recall e F1 tanto no treino quanto no teste.

Isso permite observar:
- Se o modelo sofre de underfitting (métricas ruins em ambos)
- Se sofre de overfitting (treino alto, teste baixo)
- A partir de qual volume de dados o desempenho estabiliza

In [38]:
# Reconstruir os classificadores com os melhores hiperparâmetros já encontrados
best_classifiers = {
    'Árvore de Decisão': DecisionTreeClassifier(
        criterion='entropy', max_depth=3, min_samples_leaf=1,
        min_samples_split=2, random_state=42
    ),
    'Naive Bayes': GaussianNB(
        var_smoothing=1e-07
    ),
    'Regressão Logística': LogisticRegression(
        C=0.1, class_weight=None, penalty='l2',
        solver='liblinear', max_iter=1000, random_state=42
    ),
    'KNN': KNeighborsClassifier(
        metric='euclidean', n_neighbors=15, weights='distance'
    ),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', max_depth=5, min_samples_leaf=1,
        min_samples_split=2, n_estimators=50, random_state=42
    )
}

# Proporções de treino: 5% a 95% com passo de 5%
train_fractions = np.arange(0.05, 1.0, 0.05)

metrics_names = ['Precision', 'Recall', 'F1-score']
metric_funcs = {
    'Precision': lambda yt, yp: precision_score(yt, yp, average='weighted', zero_division=0),
    'Recall': lambda yt, yp: recall_score(yt, yp, average='weighted', zero_division=0),
    'F1-score': lambda yt, yp: f1_score(yt, yp, average='weighted', zero_division=0)
}

# Armazenar resultados: {classificador: {metrica: {treino: [...], teste: [...]}}}
learning_curves = {}

for clf_name, clf in best_classifiers.items():
    learning_curves[clf_name] = {m: {'treino': [], 'teste': []} for m in metrics_names}
    
    for frac in train_fractions:
        test_frac = 1.0 - frac
        
        # Com desbalanceamento extremo (34 amostras da classe 2),
        # splits muito pequenos podem ter 0 amostras de uma classe.
        # Garantimos pelo menos 1 amostra de cada classe em cada parte.
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_cls, y_cls, train_size=frac, random_state=42, stratify=y_cls
            )
        except ValueError:
            # Se não for possível estratificar (muito poucas amostras), preenche NaN
            for m in metrics_names:
                learning_curves[clf_name][m]['treino'].append(np.nan)
                learning_curves[clf_name][m]['teste'].append(np.nan)
            continue
        
        from sklearn.base import clone
        model = clone(clf)
        model.fit(X_tr, y_tr)
        
        y_pred_tr = model.predict(X_tr)
        y_pred_te = model.predict(X_te)
        
        for m in metrics_names:
            learning_curves[clf_name][m]['treino'].append(metric_funcs[m](y_tr, y_pred_tr))
            learning_curves[clf_name][m]['teste'].append(metric_funcs[m](y_te, y_pred_te))
    
    print(f'{clf_name}: curva calculada')

print(f'\nFrações de treino: {len(train_fractions)} pontos (5% a 95%)')

Árvore de Decisão: curva calculada
Naive Bayes: curva calculada
Regressão Logística: curva calculada
KNN: curva calculada
Random Forest: curva calculada

Frações de treino: 19 pontos (5% a 95%)


## 3.1 Curvas de aprendizagem — um gráfico por métrica

Cada gráfico mostra, para uma métrica (Precision, Recall ou F1), a evolução no treino e no teste de todos os 5 classificadores conforme a fração de treino varia de 5% a 95%.

In [ ]:
colors = {
    'Árvore de Decisão': '#2563eb',
    'Naive Bayes': '#dc2626',
    'Regressão Logística': '#16a34a',
    'KNN': '#9333ea',
    'Random Forest': '#ea580c'
}

train_pcts = (train_fractions * 100).astype(int)

metric_file_num = {'Precision': '06', 'Recall': '07', 'F1-score': '08'}

for metric in metrics_names:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for clf_name in best_classifiers:
        treino_vals = learning_curves[clf_name][metric]['treino']
        teste_vals = learning_curves[clf_name][metric]['teste']
        cor = colors[clf_name]
        
        ax.plot(train_pcts, teste_vals, 'o-', color=cor, linewidth=2,
                markersize=5, label=f'{clf_name} (teste)')
        ax.plot(train_pcts, treino_vals, 's--', color=cor, linewidth=1,
                markersize=3, alpha=0.5, label=f'{clf_name} (treino)')
    
    ax.set_xlabel('Fração de treino (%)', fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'Curva de aprendizagem — {metric}', fontsize=14)
    ax.set_xticks(train_pcts)
    ax.set_xticklabels(train_pcts, rotation=45, fontsize=8)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.85, 1.005)
    
    num = metric_file_num[metric]
    plt.tight_layout()
    plt.savefig(f'results/{num}_curva_aprendizagem_{metric.lower().replace("-", "")}.png', dpi=150)
    plt.show()

## 3.2 Análise das curvas de aprendizagem

**Comportamento geral:**
A maioria dos classificadores atinge desempenho quase perfeito já com frações pequenas de treino (~10-15%). Isso se explica pela natureza do problema: os clusters gerados pelo K-Means com k*=2 produzem uma separação nítida no espaço normalizado (silhueta de 0.66), com fronteira de decisão relativamente simples.

**Árvore de Decisão, KNN e Random Forest:**
Esses três modelos convergem rapidamente para F1 ≈ 1.0 tanto no treino quanto no teste. A curva de treino e a curva de teste praticamente se sobrepõem, indicando ausência de overfitting. A árvore com profundidade máxima 3 já é suficiente para capturar a fronteira entre os clusters, o que confirma que a separação é geometricamente simples.

**Regressão Logística:**
Apresenta comportamento semelhante aos anteriores, com convergência rápida. A leve diferença em relação ao F1 perfeito (0.999 vs 1.0) reflete a dificuldade de um modelo linear em capturar perfeitamente a fronteira quando a classe minoritária (34 amostras) está muito próxima da fronteira de decisão.

**Naive Bayes:**
É o classificador com pior desempenho relativo. O recall no teste é consistentemente mais baixo (~0.93), indicando que o modelo erra mais na classe majoritária (cluster 1) por conta da suposição de independência entre features, que não se sustenta bem com 57 features correlacionadas. A curva de treino e teste não convergem completamente, sugerindo um viés intrínseco do modelo (underfitting estrutural).

**Efeito do desbalanceamento:**
Com apenas 34 amostras no cluster 2 (0.7%), as frações muito pequenas de treino (5%) disponibilizam apenas 1-2 amostras dessa classe. Mesmo assim, modelos como KNN e Random Forest conseguem classificar corretamente, graças ao uso de `weights='distance'` e `class_weight='balanced'`, respectivamente.

**Conclusão:**
A tarefa de classificação dos clusters do K-Means é relativamente fácil para a maioria dos modelos, o que é esperado — os clusters foram gerados por um algoritmo de particionamento que maximiza a separação. Naive Bayes é a exceção por suas limitações estruturais. Em cenários reais com classes naturais (e não derivadas de clustering), espera-se maior variabilidade nas curvas de aprendizagem.